# Sprinter — Phase 7 & Phase 10: Unified Multi-Agent Supervisor
### Orchestration of QA (Feature 2), DocGen Phase 1 Baseline (Feature 1), and SRS (Feature 3)

This notebook implements and demonstrates **Phase 7** (Supervisor routing & persistence) and **Phase 10** (Full 3-Way Cross-Feature Integration with Unified Shared Services Layer) from `Sprinter_Implementation_Plan.md`.

**Architecture:**
```
                         ┌─────────────────────────┐
                         │  Supervisor / Router    │
                         └────────────┬────────────┘
                  ┌───────────────────┼───────────────────┐
                  ▼                   ▼                   ▼
         ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
         │   QA Subgraph   │ │ DocGen Subgraph │ │  SRS Subgraph   │
         │   (Feature 2)   │ │  (Phase 1 Base) │ │ (HITL + Workers)│
         └─────────────────┘ └─────────────────┘ └─────────────────┘
                  │                   │                   │
                  └───────────────────┴───────────────────┘
                                      ▼
                           Unified ServiceRegistry
           (Multi-Key Gemini Failover, Tavily, Chroma DB, GitHub MCP)
```

In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Ensure project root is in sys.path
project_root = os.path.abspath("../..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core.config import config
from src.core.service_registry import services
from src.supervisor.sprinter import sprinter, SprinterAssistant

print(f"✓ Config loaded. Gemini Keys: {len(config.google_api_keys)} | Mistral: {bool(config.mistral_api_key)} | Tavily: {bool(config.tavily_api_key)}")
print(f"✓ Sprinter Assistant initialized with DocGen Engine: '{sprinter.docgen_version}'")

## 2. Test 1: Feature 2 — QA Subagent Dispatch
The Supervisor classifies the question as `qa` and routes to the 3-way QA subgraph (Direct / Web Search / Code RAG).

In [ ]:
qa_query = "What is dependency injection and how does it reduce coupling in software architecture?"
res_qa = sprinter.run(query=qa_query, thread_id="demo_qa")

print(f"Status: {res_qa.status} | Route: {res_qa.route}")
print("\n--- Output ---")
print(res_qa.output)

## 3. Test 2: Feature 1 — DocGen Subagent (Phase 1 Baseline)
The Supervisor routes documentation requests to the Phase 1 Baseline DocGen pipeline (`repo_map_generator -> doc_intent_router -> direct_tool_fetch -> baseline_docgen`).

In [ ]:
doc_query = "Document how the GitHubMCPClient handles repo tree fetching in src/github_mcp_client.py"
res_doc = sprinter.run(query=doc_query, thread_id="demo_docgen", local_path=".")

print(f"Status: {res_doc.status} | Route: {res_doc.route} | Citations: {res_doc.metadata.get('citations_count', 0)}")
print("\n--- Generated Documentation Preview ---")
print(res_doc.output[:800] + "\n...[truncated]")

## 4. Test 3: Feature 3 — SRS Subagent with HITL Pause & Resume
The Supervisor dispatches specification asks to the SRS Subagent. The agent pauses at `interrupt()` with a targeted clarification question, and resumes upon user reply.

In [ ]:
srs_thread = "interactive_srs_demo"
srs_query = "Create an IEEE 830 software requirements specification for an AI-powered telemedicine consultation app"

# Turn 1: Initial invocation
res_srs_turn1 = sprinter.run(query=srs_query, thread_id=srs_thread)
print(f"Turn 1 Status: {res_srs_turn1.status} | Route: {res_srs_turn1.route}")
if res_srs_turn1.status == "waiting_human_input":
    print(f"Pending Question for Developer:\n{res_srs_turn1.pending_question}")

In [ ]:
# Turn 2: Providing answers to the clarification question
user_response = "The app must support HIPAA-compliant encrypted video streaming, WebRTC, real-time doctor availability calendars, and automated prescription dispatch."

res_srs_turn2 = sprinter.resume(thread_id=srs_thread, user_response=user_response)
print(f"Turn 2 Status: {res_srs_turn2.status}")
if res_srs_turn2.status == "completed":
    print(f"\n✓ Final Document Generated ({len(res_srs_turn2.output)} chars)!")
    print(res_srs_turn2.output[:800] + "\n...[truncated]")
elif res_srs_turn2.status == "waiting_human_input":
    print(f"Next Clarification Question:\n{res_srs_turn2.pending_question}")

## 5. Phase 10 Cross-Feature Regression Suite
Run the full automated cross-feature regression suite across 20+ golden dataset samples and all 3 agent pipelines.

In [ ]:
from evals.eval_cross_feature_regression import run_cross_feature_regression
report = run_cross_feature_regression()
print("\nRegression Suite Overall Result:", report["overall_status"])